Data: Kaggle competition [deep-past-initiative-machine-translation](https://www.kaggle.com/competitions/deep-past-initiative-machine-translation) (Old Assyrian transliterations with English translations).
Download it into `./deep-past-initiative-machine-translation/`.

In [ ]:
import os
os.environ['HSA_OVERRIDE_GFX_VERSION']="11.0.0"
os.environ['TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL']="1"

In [ ]:
import re, math, random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

TRAIN_PATH = "./deep-past-initiative-machine-translation/train.csv"
TEST_PATH  = "./deep-past-initiative-machine-translation/test.csv"

DEVICE = "cuda"
print("Device:", DEVICE)

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("train:", train_df.shape, "test:", test_df.shape)

In [ ]:
published_texts = pd.read_csv("./deep-past-initiative-machine-translation/published_texts.csv")
published_texts.head()

In [ ]:
publications = pd.read_csv("./deep-past-initiative-machine-translation/publications.csv")

publications_akkadian_rows = publications[publications["has_akkadian"]]
publications_akkadian_rows.head()

In [ ]:
publications_akkadian_rows['page_text'].iloc[211]

In [ ]:
bibliography = pd.read_csv("./deep-past-initiative-machine-translation/bibliography.csv")
bibliography.head()

In [ ]:
OA_Lexicon_eBL = pd.read_csv("./deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv")
OA_Lexicon_eBL.head()

In [ ]:
def clean_ws(x: str) -> str:
    if pd.isna(x): return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def split_src_transliteration(doc: str):
    doc = "" if pd.isna(doc) else str(doc)
    parts = re.split(r"\n+", doc)
    parts = [clean_ws(p) for p in parts if clean_ws(p)]
    if len(parts) <= 1:
        parts = re.split(r"\s{2,}|;\s+", clean_ws(doc))
        parts = [clean_ws(p) for p in parts if clean_ws(p)]
    return parts

def split_tgt_english(doc: str):
    doc = clean_ws(doc)
    if not doc:
        return []
    sents = re.split(r"(?<=[.!?])\s+", doc)
    sents = [clean_ws(s) for s in sents if clean_ws(s)]
    return sents

def tok_len(s: str) -> int:
    return max(1, len(s.split()))

def align_monotone(src_units, tgt_units, max_src_merge=2, max_tgt_merge=2):
    """
    Monotone greedy alignment by matching length proportions.
    Produces pairs (src_text, tgt_text), allowing merges 1-2 units.
    """
    i = j = 0
    pairs = []
    while i < len(src_units) and j < len(tgt_units):
        best = None
        for si in range(1, max_src_merge+1):
            for tj in range(1, max_tgt_merge+1):
                if i+si <= len(src_units) and j+tj <= len(tgt_units):
                    src_chunk = " ".join(src_units[i:i+si])
                    tgt_chunk = " ".join(tgt_units[j:j+tj])
                    ls, lt = tok_len(src_chunk), tok_len(tgt_chunk)
                    score = abs(math.log(ls/lt))
                    if (best is None) or (score < best[0]):
                        best = (score, si, tj, src_chunk, tgt_chunk)
        if best is None:
            break
        _, si, tj, s_chunk, t_chunk = best
        pairs.append((s_chunk, t_chunk))
        i += si
        j += tj

    if pairs:
        if i < len(src_units):
            pairs[-1] = (pairs[-1][0] + " " + " ".join(src_units[i:]), pairs[-1][1])
        if j < len(tgt_units):
            pairs[-1] = (pairs[-1][0], pairs[-1][1] + " " + " ".join(tgt_units[j:]))

    return pairs

pairs = []
for _, row in train_df.iterrows():
    src_doc = row["transliteration"]
    tgt_doc = row["translation"]

    src_lines = split_src_transliteration(src_doc)
    tgt_sents = split_tgt_english(tgt_doc)

    if len(src_lines) == 0 or len(tgt_sents) == 0:
        continue

    doc_pairs = align_monotone(src_lines, tgt_sents, max_src_merge=2, max_tgt_merge=2)
    pairs.extend(doc_pairs)

pairs_df = pd.DataFrame(pairs, columns=["src", "tgt"])
pairs_df = pairs_df[(pairs_df["src"].str.len() > 0) & (pairs_df["tgt"].str.len() > 0)].drop_duplicates()

print("Sentence-level pairs:", pairs_df.shape)
pairs_df.head()

In [ ]:
def load_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained("sdadas/mt5-base-translator-pl-en")  # facebook/nllb-200-distilled-600M
    tokenizer.use_fast=False
    model = AutoModelForSeq2SeqLM.from_pretrained("sdadas/mt5-base-translator-pl-en")  # facebook/nllb-200-distilled-600M
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer()
model.to(DEVICE)

In [ ]:
MAX_SRC = 256
MAX_TGT = 128

class MTDataset(Dataset):
    def __init__(self, df):
        self.src = df["src"].tolist()
        self.tgt = df["tgt"].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        x = tokenizer(
            self.src[idx],
            max_length=MAX_SRC,
            truncation=True,
        )
        y = tokenizer(
            text_target=self.tgt[idx],
            max_length=MAX_TGT,
            truncation=True,
        )
        x["labels"] = y["input_ids"]
        return x

def preprocess(batch):
    x = tokenizer(
        batch["src"],
        max_length=MAX_SRC,
        truncation=True,
    )
    y = tokenizer(
        text_target=batch["tgt"],
        max_length=MAX_TGT,
        truncation=True,
    )
    x["labels"] = y["input_ids"]
    return x

In [ ]:
val_frac = 0.08
pairs_df = pairs_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_val = max(1, int(len(pairs_df) * val_frac))
val_df = pairs_df.iloc[:n_val].reset_index(drop=True)
trn_df = pairs_df.iloc[n_val:].reset_index(drop=True)

train_ds = MTDataset(trn_df)
val_ds   = MTDataset(val_df)

def collate_fn(features):
    proc = [preprocess(f) for f in features]
    return data_collator(proc)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir="translator_deeppast",
    eval_strategy="steps",
    eval_steps=300,
    save_steps=300,
    logging_steps=50,
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    fp16=False,
    bf16=True,
    num_train_epochs=35,
    predict_with_generate=True,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    torch_compile=True,
    optim="adamw_torch_fused"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)
trainer.train()

In [ ]:
def make_one_sentence(en: str) -> str:
    en = clean_ws(en)
    if not en:
        return "..."
    parts = re.split(r"(?<=[.!?])\s+", en)
    parts = [p.strip() for p in parts if p.strip()]
    parts = [re.sub(r"[.!?]+\s*$", "", p).strip() for p in parts]
    parts = [p for p in parts if p]
    out = "; ".join(parts).strip()
    if not out.endswith("."):
        out += "."
    return out

model.eval()

def generate_translation(src_text: str):
    enc = tokenizer(
        clean_ws(src_text),
        return_tensors="pt",
        max_length=MAX_SRC,
        truncation=True,
    ).to(DEVICE)

    with torch.no_grad():
        out_ids = model.generate(
            **enc,
            num_beams=4,
            max_new_tokens=MAX_TGT,
            length_penalty=1.0,
            early_stopping=True,
        )
    pred = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    return make_one_sentence(pred)

# preds = [generate_translation(t) for t in test_df["transliteration"].tolist()]
# sub = pd.DataFrame({"id": test_df["id"], "translation": preds})
# sub.to_csv("submission.csv", index=False)
# print("Wrote submission.csv")
# sub.head()

In [ ]:
from difflib import SequenceMatcher
from tqdm import tqdm


def perturb_sentence(sentence: str) -> str:
    words = sentence.split()
    if len(words) > 3 and random.random() < 0.8:
        random.shuffle(words)
        return " ".join(words)
    return sentence

def is_too_similar(pred: str, ref: str, threshold: float =  0.85) -> bool:
    return SequenceMatcher(None, pred, ref).ratio() > threshold

negative_rows = []

model.eval()
for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Generating negatives"):
    translit = row['transliteration']
    reference = row['translation']

    pred = generate_translation(translit)
    pred = perturb_sentence(pred)

    if not is_too_similar(pred, reference):
        negative_rows.append({
            'transliteration': translit,
            'translation': pred,
            'label': 0
        })

negative_df = pd.DataFrame(negative_rows)
negative_df.to_csv("negative_pairs.csv", index=False)
print(f"Saved {len(negative_df)} negative pairs to negative_pairs.csv")

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from tqdm import tqdm
from torch.optim import AdamW


REWARD_MODEL_NAME = "sdadas/xlm-roberta-large-twitter"
BATCH_SIZE = 4
EPOCHS = 3
LR = 2e-5

torch.manual_seed(SEED)

class TranslationPairDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        translit = str(row['transliteration'])
        translation = str(row['translation'])
        label = float(row['label'])  # 1=good, 0=bad

        encoded = self.tokenizer(
            translit,
            translation,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        input_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.float)
        }


train_pos = pd.read_csv("deep-past-initiative-machine-translation/train.csv")
train_pos['label'] = 1

train_neg = pd.read_csv("negative_pairs.csv")
train_neg['label'] = 0

train_df = pd.concat([train_pos, train_neg]).reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_NAME)
dataset = TranslationPairDataset(train_df, tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

class RewardModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:,0,:]
        x = self.dropout(cls_emb)
        x = self.classifier(x)
        score = self.sigmoid(x).squeeze(-1)
        return score

model = RewardModel(REWARD_MODEL_NAME).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR)
criterion = nn.BCELoss()


model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        scores = model(input_ids, attention_mask)
        loss = criterion(scores, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Avg Loss: {total_loss/len(dataloader):.4f}")

model.save_pretrained("reward_model_xlm_roberta")
tokenizer.save_pretrained("reward_model_xlm_roberta")

In [ ]:
%pip install rapidfuzz

In [ ]:
from rapidfuzz import fuzz, process

ocr_text = "KIŠIB a-šùr-ma-lik DUMU i-na-a a-na ..."
lexicon = OA_Lexicon_eBL['norm'].tolist()

matches = process.extract(ocr_text, lexicon, scorer=fuzz.partial_ratio, score_cutoff=80)